<a href="https://colab.research.google.com/github/tommbk50-hub/GCMS-App/blob/main/auto_hplc_gcms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Install required Python libraries
!pip -q install ipyketcher rdkit mols2grid pyteomics psims lxml hdf5plugin pandas pubchempy mendeleev marimo wigglystuff

import os
import glob
import pandas as pd

# Clone the repository to get the files from the specific directory
if not os.path.exists('hplc-py'):
    !git clone -q https://github.com/cremerlab/hplc-py.git

In [ ]:
#@title Plot HPLC Chromatogram, Convert the HPLC Trace into a Pandas dataframe
import urllib.request
import os
import matplotlib.pyplot as plt
from pyteomics import mzml
import pandas as pd

# 1. Download the new dataset
url_new = "https://raw.githubusercontent.com/OpenMS/OpenMS/develop/src/tests/topp/FeatureFinderMetaboIdent_1_input.mzML"
filename_new = "FeatureFinderMetaboIdent_1_input.mzML"

if not os.path.exists(filename_new):
    print(f"Downloading {filename_new}...")
    urllib.request.urlretrieve(url_new, filename_new)

# 2. Parse the new dataset
print(f"Parsing {filename_new}...")
rt_new = []
tic_new = []

try:
    with mzml.read(filename_new) as reader:
        for spectrum in reader:
            if spectrum['ms level'] == 1:
                scan = spectrum.get('scanList', {}).get('scan', [{}])[0]
                rt = scan.get('scan start time', 0)
                tic = spectrum.get('total ion current', 0)
                rt_new.append(rt)
                tic_new.append(tic)

    # Create a pandas DataFrame from the parsed data
    hplc_df = pd.DataFrame({
        'Signal': tic_new,
        'Time': rt_new
    })

    print("HPLC Trace DataFrame successfully created:")
    print(hplc_df.head())

    # 3. Plot the new HPLC trace using the DataFrame
    plt.figure(figsize=(10, 6))
    plt.plot(hplc_df['Time'], hplc_df['Signal'], color='green', linewidth=1.5)
    plt.title('HPLC Trace: FeatureFinderMetaboIdent_1_input.mzML')
    plt.xlabel('Retention Time (seconds)')
    plt.ylabel('Total Ion Current (Signal)')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"An error occurred: {e}")

# @title Automatically detect and fit the peaks (Chromatogram Normalization, Baseline Correction (SNIP), Topographic Peak Detection and Mixture Deconvolution) = detected_df

import scipy.signal
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import sys
import os
import numpy as np
import pandas as pd
from IPython.display import display, clear_output, Markdown
import ipywidgets as widgets
import warnings

# Suppress the hplc-py UserWarnings for a cleaner output
warnings.filterwarnings('ignore')

# Create a normalized signal using the 'Signal' column
signal_raw = hplc_df['Signal'].values
signal_norm = (signal_raw - signal_raw.min()) / (signal_raw.max() - signal_raw.min())

# --- Dynamic Hyperparameter Optimization (Elbow Method) ---
print("Running Dynamic Hyperparameter Optimization...")
if 'hplc-py' not in sys.path:
    sys.path.append(os.path.abspath('hplc-py'))
from hplc.quant import Chromatogram

prominence_range = np.logspace(-3, -1, 15)
buffer_val = 0
results = []
chrom_base = Chromatogram(hplc_df, cols={'time':'Time', 'signal':'Signal'}, time_window=[200, 300])

for p in prominence_range:
    try:
        peaks = chrom_base.fit_peaks(prominence=p, buffer=buffer_val)
        num_peaks = len(peaks)
        results.append((p, num_peaks))
    except Exception:
        pass

if not results:
    default_prominence = 0.05
    print("Optimization failed. Using default prominence: 0.05")
else:
    proms_all = np.array([x[0] for x in results])
    counts_all = np.array([x[1] for x in results])

    # 2. Autonomous "Elbow" Detection (Corrected)
    # Find the peak maximum to start the downward curve, avoiding artificial upward spikes
    max_idx = np.argmax(counts_all)

    valid_proms = proms_all[max_idx:]
    valid_counts = counts_all[max_idx:]

    if len(valid_proms) > 1:
        # Calculate the maximum geometric distance from the downward curve to the secant line
        p1 = np.array([np.log10(valid_proms[0]), valid_counts[0]])
        p2 = np.array([np.log10(valid_proms[-1]), valid_counts[-1]])

        distances = []
        for i in range(len(valid_proms)):
            p3 = np.array([np.log10(valid_proms[i]), valid_counts[i]])

            # Distance from point to line (using explicit 2D cross product to avoid NumPy 2.0 deprecation)
            if np.linalg.norm(p2 - p1) == 0:
                dist = 0
            else:
                cross_prod_2d = (p2[0] - p1[0]) * (p1[1] - p3[1]) - (p2[1] - p1[1]) * (p1[0] - p3[0])
                dist = np.abs(cross_prod_2d) / np.linalg.norm(p2 - p1)
            distances.append(dist)

        optimal_idx_relative = np.argmax(distances)
        optimal_prominence = valid_proms[optimal_idx_relative]
        optimal_count = valid_counts[optimal_idx_relative]
    else:
        optimal_prominence = valid_proms[0]
        optimal_count = valid_counts[0]

    default_prominence = optimal_prominence

    # 3. Export to Global Scope for downstream cells
    prominence_val = optimal_prominence

    display(Markdown("---"))
    display(Markdown(f"### ✅ Optimization Complete"))
    display(Markdown(f"**Optimal Prominence Threshold:** `{optimal_prominence:.4f}`"))
    display(Markdown(f"**Peaks Resolved:** `{optimal_count}`"))
    display(Markdown("*(The `prominence_val` variable has been automatically updated. You may now run the peak fitting cell.)*"))

    # 4. Render the Optimization Landscape
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [2.5, 1]})

    # Plot the entire valid curve
    ax1.plot(proms_all, counts_all, marker='o', linestyle='-', color='#2c3e50', linewidth=2, label='Fitted Peaks')

    # Highlight the optimal elbow point
    ax1.axvline(optimal_prominence, color='#e74c3c', linestyle='--', linewidth=2,
                label=f'Optimal Elbow ({optimal_prominence:.4f})')
    ax1.scatter([optimal_prominence], [optimal_count], color='#e74c3c', s=150, zorder=5)

    ax1.set_xscale('log')
    ax1.set_xticks(prominence_range)
    ax1.set_xticklabels([f"{x:.4f}" for x in prominence_range], rotation=45)
    ax1.set_title('Hyperparameter Optimization: Signal-to-Noise Ratio (L-Curve)', fontsize=14, pad=15)
    ax1.set_xlabel('Prominence Threshold (Log Scale)', fontsize=12)
    ax1.set_ylabel('Number of Deconvoluted Peaks', fontsize=12)
    ax1.legend(fontsize=11)
    ax1.grid(True, which="both", ls="--", alpha=0.4)

    # Render table to the right
    ax2.axis('tight')
    ax2.axis('off')
    table_data = [[f"{p:.4f}", int(c)] for p, c in zip(proms_all, counts_all)]
    col_labels = ['Prominence Threshold', 'Number of\nDeconvoluted Peaks']
    table = ax2.table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.1, 2)

    # Highlight the optimum elbow values row in red
    optimal_row_idx = list(proms_all).index(optimal_prominence) + 1
    for (row, col), cell in table.get_celld().items():
        if row == optimal_row_idx:
            cell.set_edgecolor('red')
            cell.set_linewidth(2)

    ax2.set_title('Tabular Data', fontsize=12, pad=15)

    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------

# Create UI widgets
prominence_input = widgets.FloatText(
    value=default_prominence,
    description='Prominence:',
    step=0.001,
    style={'description_width': 'initial'}
)

rerun_button = widgets.Button(
    description='Rerun Peak Fitting',
    button_style='primary',
    icon='refresh'
)

output_area = widgets.Output()

def run_peak_fitting(prom_val):
    global detected_df
    with output_area:
        clear_output(wait=True)
        print(f"Using prominence threshold: {prom_val:.4f}")

        # Find peaks with a prominence filter
        peak_indices, _ = scipy.signal.find_peaks(signal_norm, prominence=prom_val)
        _, left_bases, right_bases = scipy.signal.peak_prominences(signal_norm, peak_indices)

        start_times = hplc_df['Time'].iloc[left_bases].values
        end_times = hplc_df['Time'].iloc[right_bases].values

        print(f"Detected {len(peak_indices)} raw topographical anomalies.")

        # --- Detect and deconvolute HPLC Chromatogram peaks with chrom.fit_peaks ---
        print("\nDeconvoluting and filtering true peaks...")
        if 'hplc-py' not in sys.path:
            sys.path.append(os.path.abspath('hplc-py'))
        from hplc.quant import Chromatogram

        chrom = Chromatogram(hplc_df, cols={'time':'Time', 'signal':'Signal'}, time_window=[200, 300])

        # Explicitly pass the prominence_val so both algorithms are synchronized
        peaks = chrom.fit_peaks(prominence=prom_val, buffer=0).reset_index(drop=True)

        # Map start/end times and the exact topographical apex to the deconvoluted peaks
        mapped_starts = []
        mapped_ends = []
        mapped_topos = []
        peak_rts = hplc_df['Time'].values[peak_indices]

        for rt in peaks['retention_time']:
            # Find the closest raw topographical peak to the fitted Gaussian center
            if len(peak_rts) > 0:
                closest_idx = np.argmin(np.abs(peak_rts - rt))
                mapped_starts.append(start_times[closest_idx])
                mapped_ends.append(end_times[closest_idx])
                mapped_topos.append(peak_rts[closest_idx])
            else:
                 mapped_starts.append(np.nan)
                 mapped_ends.append(np.nan)
                 mapped_topos.append(np.nan)

        peaks['peak start time'] = mapped_starts
        peaks['peak end time'] = mapped_ends
        peaks['topographical_apex'] = mapped_topos

        # Add normalised area column as a percentage
        if peaks['area'].sum() > 0:
            peaks['normalised area'] = (peaks['area'] / peaks['area'].sum() * 100).map('{:.2f}%'.format)
        else:
             peaks['normalised area'] = '0.00%'

        # --- Extract top 10 dominant m/z signals for each peak window ---
        from pyteomics import mzml

        mzml_filename = "FeatureFinderMetaboIdent_1_input.mzML"
        if not os.path.exists(mzml_filename):
            mzml_filename = "/content/FeatureFinderMetaboIdent_1_input.mzML"

        peak_spectra = {i: {} for i in range(len(peaks))}

        if os.path.exists(mzml_filename):
            with mzml.read(mzml_filename) as reader:
                for spectrum in reader:
                    if spectrum['ms level'] == 1:
                        scan_data = spectrum.get('scanList', {}).get('scan', [{}])
                        rt = -1
                        if isinstance(scan_data, list) and len(scan_data) > 0:
                            rt = scan_data[0].get('scan start time', -1)
                        elif isinstance(scan_data, dict):
                            rt = scan_data.get('scan start time', -1)

                        mz_array = spectrum.get('m/z array', [])
                        int_array = spectrum.get('intensity array', [])

                        if len(mz_array) > 0:
                            mzs_rounded = np.round(mz_array, 3)
                            for i, row in peaks.iterrows():
                                if not pd.isna(row['peak start time']) and row['peak start time'] <= rt <= row['peak end time']:
                                    for m, intensity in zip(mzs_rounded, int_array):
                                        peak_spectra[i][m] = peak_spectra[i].get(m, 0) + intensity

        TOP_N_PEAKS = 10
        MIN_REL_INTENSITY = 2.0

        dominant_mz_col = []
        for i in range(len(peaks)):
            spectra = peak_spectra[i]
            if not spectra:
                dominant_mz_col.append("None")
                continue

            sorted_mzs = sorted(spectra.items(), key=lambda x: x[1], reverse=True)
            max_int = sorted_mzs[0][1]

            top_strs = []
            for m, inten in sorted_mzs:
                rel_int = (inten / max_int) * 100
                if rel_int >= MIN_REL_INTENSITY:
                    top_strs.append(f"{m:.3f} ({rel_int:.0f}%)")
                if len(top_strs) >= TOP_N_PEAKS:
                    break

            dominant_mz_col.append(", ".join(top_strs) if top_strs else "None")

        peaks['dominant m/z'] = dominant_mz_col
        # --------------------------------------------------------------------------------

        detected_df = peaks.copy()

        def highlight_peaks(row):
            colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
            c = mcolors.to_rgba(colors[(int(row['peak_id']) - 1) % len(colors)], alpha=0.4)
            bg_color = f'rgba({int(c[0]*255)}, {int(c[1]*255)}, {int(c[2]*255)}, {c[3]})'
            return [f'background-color: {bg_color}' for _ in row]

        display(peaks.style.apply(highlight_peaks, axis=1))

        print(f"\nFinal deconvoluted peak count: {len(peaks)}")

        # Visualize the combined result
        fig, ax = chrom.show()
        fig.set_size_inches(12, 7)

        # --- FIX: Plot the apex for every peak listed in the table ---
        if len(peaks) > 0:
            ax.vlines(peaks['retention_time'], 0, signal_raw.max() * 1.05, linestyle='--', color='dodgerblue', label='Topographical Apex')

        ax.set_title(f'HPLC Chromatogram Deconvolution (Prominence: {prom_val:.4f})')
        ax.set_xlabel('Retention Time (seconds)')
        ax.set_ylabel('Total Ion Count (baseline corrected)')

        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles, labels, bbox_to_anchor=(1.05, 1), loc='upper left')

        plt.tight_layout()
        plt.show()

def on_rerun_clicked(b):
    run_peak_fitting(prominence_input.value)

rerun_button.on_click(on_rerun_clicked)

# Display UI and run initially with the default value
display(widgets.HBox([prominence_input, rerun_button]))
display(output_area)

run_peak_fitting(default_prominence)

In [ ]:
#@title Mass Spectrometry Fragmentation Tree:

# --- PATCH IPYKETCHER BUG ---
import sys, os
ipk_init = f"/usr/local/lib/python{sys.version_info.major}.{sys.version_info.minor}/dist-packages/ipyketcher/__init__.py"
if os.path.exists(ipk_init):
    with open(ipk_init, 'r') as f:
        content = f.read()
    if "traitlets.List[str]()" in content:
        with open(ipk_init, 'w') as f:
            f.write(content.replace("traitlets.List[str]()", "traitlets.List()"))
# ----------------------------

#imports
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import Recap, Descriptors, BRICS, AllChem, Draw
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.rdMolDescriptors import CalcMolFormula
from IPython.display import display, HTML
import ipywidgets as w, itertools, re
import pandas as pd
from ipyketcher import KetcherEditor
from google.colab import data_table
from collections import Counter
import pubchempy as pc
import io

# Comprehensive list of common MS adducts
adducts_data = [
    {"Adduct": "[M+H]+", "Ion Mode": "Positive", "Mass Shift (Da)": 1.007276, "Charge": 1},
    {"Adduct": "[M+Li]+", "Ion Mode": "Positive", "Mass Shift (Da)": 7.016005, "Charge": 1},
    {"Adduct": "[M+NH4]+", "Ion Mode": "Positive", "Mass Shift (Da)": 18.033823, "Charge": 1},
    {"Adduct": "[M+Na]+", "Ion Mode": "Positive", "Mass Shift (Da)": 22.989218, "Charge": 1},
    {"Adduct": "[M+K]+", "Ion Mode": "Positive", "Mass Shift (Da)": 38.963158, "Charge": 1},
    {"Adduct": "[M+CH3OH+H]+", "Ion Mode": "Positive", "Mass Shift (Da)": 33.033489, "Charge": 1},
    {"Adduct": "[M+ACN+H]+", "Ion Mode": "Positive", "Mass Shift (Da)": 42.033823, "Charge": 1},
    {"Adduct": "[M+ACN+Na]+", "Ion Mode": "Positive", "Mass Shift (Da)": 64.015765, "Charge": 1},
    {"Adduct": "[M+DMSO+H]+", "Ion Mode": "Positive", "Mass Shift (Da)": 79.021216, "Charge": 1},
    {"Adduct": "[M+IsoProp+H]+", "Ion Mode": "Positive", "Mass Shift (Da)": 61.064791, "Charge": 1},
    {"Adduct": "[M+CH3OH+Na]+", "Ion Mode": "Positive", "Mass Shift (Da)": 55.015433, "Charge": 1},
    {"Adduct": "[M+2K-H]+", "Ion Mode": "Positive", "Mass Shift (Da)": 76.91904, "Charge": 1},
    {"Adduct": "[M+2H]2+", "Ion Mode": "Positive", "Mass Shift (Da)": 2.014552, "Charge": 2},
    {"Adduct": "[M+H+NH4]2+", "Ion Mode": "Positive", "Mass Shift (Da)": 19.041099, "Charge": 2},
    {"Adduct": "[M+H+Na]2+", "Ion Mode": "Positive", "Mass Shift (Da)": 23.996494, "Charge": 2},
    {"Adduct": "[M+H+K]2+", "Ion Mode": "Positive", "Mass Shift (Da)": 39.970434, "Charge": 2},
    {"Adduct": "[M+2Na]2+", "Ion Mode": "Positive", "Mass Shift (Da)": 45.978436, "Charge": 2},
    {"Adduct": "[M+ACN+2H]2+", "Ion Mode": "Positive", "Mass Shift (Da)": 43.041099, "Charge": 2},
    {"Adduct": "[M+2ACN+2H]2+", "Ion Mode": "Positive", "Mass Shift (Da)": 84.067646, "Charge": 2},
    {"Adduct": "[M+3H]3+", "Ion Mode": "Positive", "Mass Shift (Da)": 3.021828, "Charge": 3},
    {"Adduct": "[M+2H+Na]3+", "Ion Mode": "Positive", "Mass Shift (Da)": 25.003770, "Charge": 3},
    {"Adduct": "[2M+H]+", "Ion Mode": "Positive", "Mass Shift (Da)": 1.007276, "Charge": 1},
    {"Adduct": "[2M+NH4]+", "Ion Mode": "Positive", "Mass Shift (Da)": 18.033823, "Charge": 1},
    {"Adduct": "[2M+Na]+", "Ion Mode": "Positive", "Mass Shift (Da)": 22.989218, "Charge": 1},
    {"Adduct": "[2M+K]+", "Ion Mode": "Positive", "Mass Shift (Da)": 38.963158, "Charge": 1},
    {"Adduct": "[2M+ACN+H]+", "Ion Mode": "Positive", "Mass Shift (Da)": 42.033825, "Charge": 1},
    {"Adduct": "[2M+ACN+Na]+", "Ion Mode": "Positive", "Mass Shift (Da)": 64.015767, "Charge": 1},
    {"Adduct": "[M]•+", "Ion Mode": "Positive", "Mass Shift (Da)": 0.0, "Charge": 1},

    # Negative Ion Mode
    {"Adduct": "[M-H]-", "Ion Mode": "Negative", "Mass Shift (Da)": -1.007276, "Charge": -1},
    {"Adduct": "[M+F]-", "Ion Mode": "Negative", "Mass Shift (Da)": 18.998403, "Charge": -1},
    {"Adduct": "[M+Cl]-", "Ion Mode": "Negative", "Mass Shift (Da)": 34.969402, "Charge": -1},
    {"Adduct": "[M+Br]-", "Ion Mode": "Negative", "Mass Shift (Da)": 78.918885, "Charge": -1},
    {"Adduct": "[M+HCOO]- (Formate)", "Ion Mode": "Negative", "Mass Shift (Da)": 44.998201, "Charge": -1},
    {"Adduct": "[M+FA-H]-", "Ion Mode": "Negative", "Mass Shift (Da)": 44.998201, "Charge": -1},
    {"Adduct": "[M+CH3COO]- (Acetate)", "Ion Mode": "Negative", "Mass Shift (Da)": 59.013851, "Charge": -1},
    {"Adduct": "[M+Hac-H]-", "Ion Mode": "Negative", "Mass Shift (Da)": 59.013851, "Charge": -1},
    {"Adduct": "[M+TFA-H]-", "Ion Mode": "Negative", "Mass Shift (Da)": 112.985586, "Charge": -1},
    {"Adduct": "[M-H2O-H]-", "Ion Mode": "Negative", "Mass Shift (Da)": -19.017841, "Charge": -1},
    {"Adduct": "[M+Na-2H]-", "Ion Mode": "Negative", "Mass Shift (Da)": 20.974666, "Charge": -1},
    {"Adduct": "[M+K-2H]-", "Ion Mode": "Negative", "Mass Shift (Da)": 36.948606, "Charge": -1},
    {"Adduct": "[M-2H]2-", "Ion Mode": "Negative", "Mass Shift (Da)": -2.014552, "Charge": -2},
    {"Adduct": "[M-3H]3-", "Ion Mode": "Negative", "Mass Shift (Da)": -3.021828, "Charge": -3},
    {"Adduct": "[2M-H]-", "Ion Mode": "Negative", "Mass Shift (Da)": -1.007276, "Charge": -1},
    {"Adduct": "[2M+HCOO]-", "Ion Mode": "Negative", "Mass Shift (Da)": 44.998201, "Charge": -1},
    {"Adduct": "[2M+FA-H]-", "Ion Mode": "Negative", "Mass Shift (Da)": 44.998201, "Charge": -1},
    {"Adduct": "[2M+CH3COO]-", "Ion Mode": "Negative", "Mass Shift (Da)": 59.013851, "Charge": -1},
    {"Adduct": "[2M+Hac-H]-", "Ion Mode": "Negative", "Mass Shift (Da)": 59.013851, "Charge": -1},
    {"Adduct": "[3M-H]-", "Ion Mode": "Negative", "Mass Shift (Da)": -1.007276, "Charge": -1}
]

# Hardcoded list of common isotopes to avoid repeatedly calling the mendeleev database
isotopes_data = [
    {"Element": "Hydrogen (H)", "Isotope": "1H", "Mass": 1.007825, "Natural Abundance (%)": 99.9885},
    {"Element": "Hydrogen (H)", "Isotope": "2H", "Mass": 2.014102, "Natural Abundance (%)": 0.0115},
    {"Element": "Carbon (C)", "Isotope": "12C", "Mass": 12.000000, "Natural Abundance (%)": 98.93},
    {"Element": "Carbon (C)", "Isotope": "13C", "Mass": 13.003355, "Natural Abundance (%)": 1.07},
    {"Element": "Nitrogen (N)", "Isotope": "14N", "Mass": 14.003074, "Natural Abundance (%)": 99.636},
    {"Element": "Nitrogen (N)", "Isotope": "15N", "Mass": 15.000109, "Natural Abundance (%)": 0.364},
    {"Element": "Oxygen (O)", "Isotope": "16O", "Mass": 15.994915, "Natural Abundance (%)": 99.757},
    {"Element": "Oxygen (O)", "Isotope": "17O", "Mass": 16.999131, "Natural Abundance (%)": 0.038},
    {"Element": "Oxygen (O)", "Isotope": "18O", "Mass": 17.999159, "Natural Abundance (%)": 0.205},
    {"Element": "Phosphorus (P)", "Isotope": "31P", "Mass": 30.973762, "Natural Abundance (%)": 100.0},
    {"Element": "Sulfur (S)", "Isotope": "32S", "Mass": 31.972071, "Natural Abundance (%)": 94.99},
    {"Element": "Sulfur (S)", "Isotope": "33S", "Mass": 32.971459, "Natural Abundance (%)": 0.75},
    {"Element": "Sulfur (S)", "Isotope": "34S", "Mass": 33.967867, "Natural Abundance (%)": 4.25},
    {"Element": "Sulfur (S)", "Isotope": "36S", "Mass": 35.967081, "Natural Abundance (%)": 0.01},
    {"Element": "Fluorine (F)", "Isotope": "19F", "Mass": 18.998403, "Natural Abundance (%)": 100.0},
    {"Element": "Chlorine (Cl)", "Isotope": "35Cl", "Mass": 34.968853, "Natural Abundance (%)": 75.76},
    {"Element": "Chlorine (Cl)", "Isotope": "37Cl", "Mass": 36.965903, "Natural Abundance (%)": 24.24},
    {"Element": "Bromine (Br)", "Isotope": "79Br", "Mass": 78.918337, "Natural Abundance (%)": 50.69},
    {"Element": "Bromine (Br)", "Isotope": "81Br", "Mass": 80.916290, "Natural Abundance (%)": 49.31},
    {"Element": "Iodine (I)", "Isotope": "127I", "Mass": 126.904473, "Natural Abundance (%)": 100.0},
    {"Element": "Sodium (Na)", "Isotope": "23Na", "Mass": 22.989769, "Natural Abundance (%)": 100.0},
    {"Element": "Potassium (K)", "Isotope": "39K", "Mass": 38.963706, "Natural Abundance (%)": 93.258},
    {"Element": "Potassium (K)", "Isotope": "40K", "Mass": 39.963998, "Natural Abundance (%)": 0.0117},
    {"Element": "Potassium (K)", "Isotope": "41K", "Mass": 40.961825, "Natural Abundance (%)": 6.7302},
    {"Element": "Calcium (Ca)", "Isotope": "40Ca", "Mass": 39.962591, "Natural Abundance (%)": 96.941},
    {"Element": "Calcium (Ca)", "Isotope": "42Ca", "Mass": 41.958618, "Natural Abundance (%)": 0.647},
    {"Element": "Calcium (Ca)", "Isotope": "43Ca", "Mass": 42.958766, "Natural Abundance (%)": 0.135},
    {"Element": "Calcium (Ca)", "Isotope": "44Ca", "Mass": 43.955482, "Natural Abundance (%)": 2.086},
    {"Element": "Calcium (Ca)", "Isotope": "46Ca", "Mass": 45.953689, "Natural Abundance (%)": 0.004},
    {"Element": "Calcium (Ca)", "Isotope": "48Ca", "Mass": 47.952523, "Natural Abundance (%)": 0.187},
    {"Element": "Magnesium (Mg)", "Isotope": "24Mg", "Mass": 23.985042, "Natural Abundance (%)": 78.99},
    {"Element": "Magnesium (Mg)", "Isotope": "25Mg", "Mass": 24.985837, "Natural Abundance (%)": 10.00},
    {"Element": "Magnesium (Mg)", "Isotope": "26Mg", "Mass": 25.982593, "Natural Abundance (%)": 11.01}
]

#helpers
def _smiles2mol(smi, add_hs=False):
    m = Chem.MolFromSmiles(smi)
    return Chem.AddHs(m) if add_hs else m

#recap fragmentation
def _recap_levels(mol):
    root   = Recap.RecapDecompose(mol)
    roots  = [root] if root.smiles else list(root.children.values())
    levels = []
    def walk(node, d=0):
        if len(levels) <= d:
            levels.append([])
        levels[d].append(node.smiles)
        for ch in node.children.values():
            walk(ch, d+1)
    for n in roots: walk(n, 0)
    return levels

#BRICS fragmentation
def _brics_levels(mol):
    frags = BRICS.BRICSDecompose(mol)
    return [list(frags)] if frags else [[Chem.MolToSmiles(mol)]]

#MS-style fragmentation
def _cleavable_bonds(mol):
    hetero = lambda a: a.GetAtomicNum() not in (1, 6)
    cleav  = set()
    for bond in mol.GetBonds():
        if bond.GetBondType() != Chem.BondType.SINGLE:   continue
        i, j = bond.GetBeginAtom(), bond.GetEndAtom()
        if hetero(i) or hetero(j):
            cleav.add(bond.GetIdx()); continue
        alpha = lambda at: any(hetero(n) for n in at.GetNeighbors())
        if alpha(i) or alpha(j):
            cleav.add(bond.GetIdx()); continue
        if bond.IsInRing():
            cleav.add(bond.GetIdx())
    return list(cleav)

def _ms_levels(mol, *, max_depth=3, min_heavy=2):
    levels, frontier = [[Chem.MolToSmiles(mol)]], [mol]
    seen = set(levels[0])
    for _ in range(max_depth):
        nxt = []
        for m in frontier:
            for b in _cleavable_bonds(m):
                frag  = Chem.FragmentOnBonds(m, [b], addDummies=False)
                parts = Chem.GetMolFrags(frag, asMols=True, sanitizeFrags=True)
                for p in parts:
                    if p.GetNumHeavyAtoms() < min_heavy:   continue
                    smi = Chem.MolToSmiles(p)
                    if smi not in seen:
                        seen.add(smi); nxt.append(p)
        if not nxt: break
        levels.append([Chem.MolToSmiles(x) for x in nxt])
        frontier = nxt
    return levels

# CSV-based "fragmentation"
def _csv_levels(mol):
    csv_path = '/content/mass_spectrum_peaks.csv'
    if os.path.exists(csv_path):
        try:
            df_csv = pd.read_csv(csv_path)
            smiles_list = [smi for smi in df_csv['SMILES'].dropna().unique() if str(smi).strip()]
            return [smiles_list]
        except Exception as e:
            print(f"⚠️ Error reading CSV: {e}")
    return [[Chem.MolToSmiles(mol)]]

def draw_frag_tree(smiles, *, digits=2, cell=(260,260), levels_fn=_recap_levels, ion_mode='Both'):
    parent  = _smiles2mol(smiles)
    if parent is None: raise ValueError("Invalid SMILES from Ketcher")

    parent_components = [Chem.MolToSmiles(p) for p in Chem.GetMolFrags(parent, asMols=True)]
    parent_mols = [Chem.MolFromSmiles(p) for p in parent_components]

    has_advanced_data = 'adducts_data' in globals() and 'isotopes_data' in globals()

    # 1. Collect all unique SMILES globally purely to draw the visual SVG grid
    unique_smiles_all = set()
    for p_mol in parent_mols:
        if p_mol:
            levels = levels_fn(p_mol)
            unique_smiles_all.update(set(itertools.chain.from_iterable(levels)))

    all_frags = []
    for smi in unique_smiles_all:
        m = _smiles2mol(smi)
        if m is not None:
            exact_mass = Descriptors.ExactMolWt(m)
            base_mz = exact_mass + 1.007276
            all_frags.append((smi, m, base_mz))

    all_frags.sort(key=lambda x: x[2], reverse=True)
    mols, legs = [], []
    for smi, m, mz in all_frags:
        mols.append(m)
        legs.append(f"m/z={mz:.{digits}f}")

    n_cols = max(1, int(len(mols)**0.5))
    pad = n_cols * max(1, len(mols)//n_cols) - len(mols)
    if pad > 0:
        mols.extend([Chem.Mol()] * pad)
        legs.extend([""] * pad)

    w_cell, h_cell = cell
    drawer = rdMolDraw2D.MolDraw2DSVG(n_cols*w_cell, (max(1, len(mols)//n_cols))*h_cell, w_cell, h_cell)
    drawer.drawOptions().legendFontSize = 18
    drawer.DrawMolecules(tuple(mols), legends=tuple(legs))
    drawer.FinishDrawing()
    with open('frag_tree.svg', 'w') as f: f.write(drawer.GetDrawingText())

    # 2. Build df_data TARGET BY TARGET to prevent cross-contamination of intact/frag labels
    df_data = []

    if has_advanced_data:
        isotope_dict = {}
        for iso in isotopes_data:
            match = re.search(r'\((.*?)\)', iso['Element'])
            if match:
                sym = match.group(1)
                if sym not in isotope_dict:
                    isotope_dict[sym] = []
                isotope_dict[sym].append(iso)
        for sym in isotope_dict:
            isotope_dict[sym].sort(key=lambda x: x['Natural Abundance (%)'], reverse=True)

    for i, p_mol in enumerate(parent_mols):
        if not p_mol: continue
        target_num = str(i+1)
        p_smi = parent_components[i]

        levels = levels_fn(p_mol)
        unique_smi_for_p = set(itertools.chain.from_iterable(levels))

        for smi in unique_smi_for_p:
            m = _smiles2mol(smi)
            if m is None: continue

            exact_mass = Descriptors.ExactMolWt(m)
            formula = CalcMolFormula(m)
            is_intact = (smi == p_smi)

            if has_advanced_data:
                elem_counts = Counter([atom.GetSymbol() for atom in m.GetAtoms()])
                for adduct_dict in adducts_data:
                    if ion_mode != 'Both' and adduct_dict['Ion Mode'] != ion_mode:
                        continue

                    adduct_name = adduct_dict['Adduct']
                    if not is_intact and ('[2M' in adduct_name or '[3M' in adduct_name):
                        continue

                    shift = adduct_dict['Mass Shift (Da)']
                    charge_val = adduct_dict['Charge']
                    ion_mode_str = adduct_dict['Ion Mode']
                    charge = abs(charge_val) if charge_val != 0 else 1
                    mult = 1
                    match = re.search(r'\[(\d+)M', adduct_name)
                    if match: mult = int(match.group(1))

                    base_mz = (exact_mass * mult + shift) / charge
                    label = f"T({target_num}) {adduct_name}" if is_intact else f"T({target_num}) Frag [{formula}] {adduct_name}"

                    df_data.append([target_num, smi, label, ion_mode_str, charge_val, round(base_mz, 4), 100.00])

                    for sym, count in elem_counts.items():
                        isos = isotope_dict.get(sym, [])
                        if len(isos) > 1:
                            base_iso = isos[0]
                            for minor_iso in isos[1:]:
                                minor_abund = minor_iso['Natural Abundance (%)']
                                base_abund = base_iso['Natural Abundance (%)']
                                if minor_abund > 0 and base_abund > 0:
                                    rel_abund = (count * mult) * (minor_abund / base_abund) * 100
                                    if rel_abund >= 0.1:
                                        mass_shift = minor_iso['Mass'] - base_iso['Mass']
                                        iso_mz = (exact_mass * mult + shift + mass_shift) / charge
                                        df_data.append([target_num, smi, f"{label} (+1 {minor_iso['Isotope']})", ion_mode_str, charge_val, round(iso_mz, 4), round(rel_abund, 2)])
            else:
                c12_abund, c13_abund, c13_diff = 98.93, 1.07, 1.00335
                num_c = formula.count('C') if 'C' in formula else 0
                p_all_12c = (c12_abund / 100.0) ** num_c
                p_one_13c = 1.0 - p_all_12c
                rel_abund_base = p_all_12c * 100.0
                rel_abund_13c = p_one_13c * 100.0

                if ion_mode == 'Negative':
                    ion_mode_str = "Negative"
                    charge_val = -1
                    base_mz = exact_mass - 1.007276
                    label = f"T({target_num}) [M-H]-" if is_intact else f"T({target_num}) Frag [{formula}] [M-H]-"
                    df_data.append([target_num, smi, label, ion_mode_str, charge_val, round(base_mz, 4), round(rel_abund_base, 2)])
                    df_data.append([target_num, smi, f"{label} (+13C)", ion_mode_str, charge_val, round((base_mz + c13_diff), 4), round(rel_abund_13c, 2)])
                else:
                    ion_mode_str = "Positive"
                    charge_val = 1
                    base_mz = exact_mass + 1.007276
                    label = f"T({target_num}) [M+H]+" if is_intact else f"T({target_num}) Frag [{formula}] [M+H]+"
                    df_data.append([target_num, smi, label, ion_mode_str, charge_val, round(base_mz, 4), round(rel_abund_base, 2)])
                    df_data.append([target_num, smi, f"{label} (+13C)", ion_mode_str, charge_val, round((base_mz + c13_diff), 4), round(rel_abund_13c, 2)])

    return pd.DataFrame(df_data, columns=['Target number', 'SMILES', 'Peak Name', 'Ion Mode', 'Charge', 'm/z', 'Relative Abundance (%)']).drop_duplicates(subset=['Target number', 'SMILES', 'm/z']).reset_index(drop=True)

# Initialize blank ketcher instance
ketcher = KetcherEditor()

mode_box = w.Dropdown(
    options={'Retrosynthetic (RECAP)': _recap_levels,
             'MS-like (hetero/\u03b1/ring)': _ms_levels,
             'BRICS (Building Blocks)': _brics_levels,
             'From CSV (mass_spectrum_peaks.csv)': _csv_levels},
    value=_ms_levels, description='Mode:', layout=w.Layout(width='300px'))

ion_mode_box = w.Dropdown(
    options={'Both (ESI+ / ESI-)': 'Both',
             'Positive (ESI+)': 'Positive',
             'Negative (ESI-)': 'Negative'},
    value='Both', description='Ion Mode:', layout=w.Layout(width='250px'))

btn = w.Button(description='Generate tree', button_style='success')
add_target_btn = w.Button(description='Add Drawn Target', button_style='primary', icon='plus', layout=w.Layout(width='200px'))
out = w.Output()

# Global variables
df_tree = pd.DataFrame()
searched_molecules = []  # List of dictionaries for accumulated searches

search_input = w.Textarea(placeholder="Compound names, CAS, or SMILES... (separated by commas or newlines)", description="Molecules:", layout=w.Layout(width='400px', height='80px'))
search_btn = w.Button(description="Search PubChem", button_style="info")
search_out = w.Output()
image_gallery = w.HBox(layout=w.Layout(overflow_x='auto', margin='10px 0')) # Scrollable gallery for rendered SVGs

def refresh_gallery():
    widgets = []
    for item in searched_molecules:
        img_widget = w.HTML(value=item["img_svg"], layout=w.Layout(width='150px', height='150px'))
        lbl = w.Label(value=item["name"][:20], layout=w.Layout(justify_content='center'))
        widgets.append(w.VBox([img_widget, lbl], layout=w.Layout(align_items='center', margin='0 10px')))
    image_gallery.children = tuple(widgets)

def on_search(b):
    search_out.clear_output()
    with search_out:
        RDLogger.DisableLog('rdApp.*')

        raw_q = search_input.value.strip()
        if not raw_q:
            print("Please enter Compound Names, CAS, or SMILES.")
            RDLogger.EnableLog('rdApp.*')
            return

        queries = [q.strip() for q in re.split(r'[,\n]+', raw_q) if q.strip()]

        for q in queries:
            mol = Chem.MolFromSmiles(q)
            if mol:
                smi_to_add = q
                name = "SMILES Input"
            else:
                print(f"⏳ Searching PubChem for '{q}'...")
                try:
                    results = pc.get_compounds(q, 'name')
                    if results:
                        smi_to_add = getattr(results[0], 'smiles', None)
                        if not smi_to_add:
                            smi_to_add = getattr(results[0], 'canonical_smiles', '')
                        name = results[0].iupac_name or q
                    else:
                        print(f"❌ No matches found in PubChem for '{q}'.")
                        continue
                except Exception as e:
                    print(f"❌ Error during search for '{q}': {e}")
                    continue

            mol = Chem.MolFromSmiles(smi_to_add)
            if mol:
                try:
                    AllChem.Compute2DCoords(mol)
                    drawer = rdMolDraw2D.MolDraw2DSVG(150, 150)
                    drawer.DrawMolecule(mol)
                    drawer.FinishDrawing()
                    svg = drawer.GetDrawingText()

                    searched_molecules.append({
                        "name": name,
                        "smiles": smi_to_add,
                        "img_svg": svg
                    })
                    print(f"✅ Added '{name}' to the target list.")
                    refresh_gallery()
                except Exception as e:
                    print(f"⚠️ Could not generate image for molecule: {e}")
            else:
                print(f"⚠️ Retrieved SMILES is invalid for '{q}'.")

        RDLogger.EnableLog('rdApp.*')

search_btn.on_click(on_search)

def on_add_target(b):
    search_out.clear_output()
    with search_out:
        smi_ketcher = getattr(ketcher, 'smiles', '')
        if not smi_ketcher or not smi_ketcher.strip():
            print("⚠️ Ketcher pad is empty. Draw a structure first before adding.")
            return

        mol = Chem.MolFromSmiles(smi_ketcher)
        if mol:
            try:
                AllChem.Compute2DCoords(mol)
                drawer = rdMolDraw2D.MolDraw2DSVG(150, 150)
                drawer.DrawMolecule(mol)
                drawer.FinishDrawing()
                svg = drawer.GetDrawingText()

                if not any(d.get("smiles") == smi_ketcher for d in searched_molecules):
                    searched_molecules.append({
                        "name": "Custom Structure",
                        "smiles": smi_ketcher,
                        "img_svg": svg
                    })
                    print("✅ Added drawn structure to the target list.")
                    refresh_gallery()
                else:
                    print("ℹ️ This structure is already in the target list.")

                ketcher.smiles = ''
            except Exception as e:
                print(f"⚠️ Could not process drawn structure: {e}")
        else:
            print("⚠️ Drawn structure is invalid.")

add_target_btn.on_click(on_add_target)

def _run(_):
    global df_tree
    out.clear_output()
    with out:
        try:
            combined_smiles_list = []

            smi_ketcher = getattr(ketcher, 'smiles', '')
            if smi_ketcher and smi_ketcher.strip():
                combined_smiles_list.append(smi_ketcher.strip())

            for mol_dict in searched_molecules:
                combined_smiles_list.append(mol_dict["smiles"])

            combined_smiles_list = list(set(combined_smiles_list))

            if not combined_smiles_list:
                print("⚠️ Sketch a molecule or use the PubChem search first.")
                return

            smi = ".".join(combined_smiles_list)

            mol = Chem.MolFromSmiles(smi)
            if mol:
                components = Chem.GetMolFrags(mol, asMols=True)
                print(f"Fragmenting {len(components)} distinct starting molecule(s).")
                for i, comp in enumerate(components):
                    exact_mass = Descriptors.ExactMolWt(comp)
                    print(f"Target {i+1} Exact Mass: {exact_mass:.4f} Da")
            else:
                print("Exact Mass: Could not calculate (invalid SMILES generated)")

            df_tree = draw_frag_tree(smi, digits=4, levels_fn=mode_box.value, ion_mode=ion_mode_box.value)

            is_target = df_tree['Peak Name'].astype(str).str.startswith('T(')
            is_frag = df_tree['Peak Name'].astype(str).str.contains('Frag')
            df_tree['Is_Frag'] = is_target & is_frag

            df_tree = df_tree.sort_values(by=['Is_Frag', 'm/z']).drop(columns=['Is_Frag']).reset_index(drop=True)

            df_tree.to_csv('frag_tree.csv', index=False)
            print("\n✅ Saved fragmentation tree data to global DataFrame 'df_tree'")
            display(data_table.DataTable(df_tree, include_index=False, num_rows_per_page=15))
        except Exception as e: print('⚠️', e)

btn.on_click(_run)

display(HTML("<style>.widget-output, .output_svg {background:white !important;}</style>"))

display(w.VBox([
    w.HBox([search_input, search_btn]),
    search_out,
    image_gallery,
    w.HTML("<b>Or draw custom structures below:</b>"),
    ketcher,
    add_target_btn,
    w.HBox([mode_box, ion_mode_box, btn]),
    out
]))

In [ ]:
#@title Assign, Verify & Refine Mass Spectra Signals (Ultra-Fast AI Loop & Comprehensive Plots)
import concurrent.futures
import re
import getpass
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import scipy.signal
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
from google import genai
from google.genai import types
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import Draw
import os
import textwrap
import warnings
from pyteomics import mzml
import time
import sys

# Suppress tight_layout warnings that trigger on tall tables
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib.figure')

# --- OPTIMIZED CONFIGURATION ---
MAX_WORKERS = 16
TOLERANCE_DA = 0.05
NOISE_THRESHOLD = 0.01
# -------------------------------

if 'detected_df' not in globals() or 'df_tree' not in globals() or 'hplc_df' not in globals():
    print("Error: Required dataframes (detected_df, df_tree, hplc_df) are missing. Please run previous cells first.")
else:
    start_time = time.time()

    # 1. Setup API and Model
    if 'gemini_api_key' not in globals() or not gemini_api_key:
        print("Please enter your Gemini API Key manually:")
        gemini_api_key = getpass.getpass("API Key: ")

    try:
        client = genai.Client(api_key=gemini_api_key)

        available_models = [m.name.replace('models/', '') for m in client.models.list()]
        flash_models = [name for name in available_models if re.match(r'^gemini-\d+\.\d+-flash$', name)]
        flash_models.sort()
        active_model = flash_models[-1] if flash_models else 'gemini-2.5-flash'

        print(f"🤖 Initializing Ultra-Fast Assignment Pipeline using: {active_model}\n")

        search_config = types.GenerateContentConfig(temperature=0.1)

        mzml_filename = "FeatureFinderMetaboIdent_1_input.mzML"
        if not os.path.exists(mzml_filename):
            mzml_filename = "/content/FeatureFinderMetaboIdent_1_input.mzML"

        if 'global_mzml_cache' not in globals():
            print("📥 Loading and caching .mzML data into RAM...")
            global_mzml_cache = []
            if os.path.exists(mzml_filename):
                with mzml.read(mzml_filename) as reader:
                    for spectrum in reader:
                        if spectrum['ms level'] == 1:
                            scan_data = spectrum.get('scanList', {}).get('scan', [{}])
                            rt = scan_data[0].get('scan start time', -1) if isinstance(scan_data, list) and len(scan_data) > 0 else scan_data.get('scan start time', -1)
                            global_mzml_cache.append({
                                'rt': rt,
                                'mzs': np.round(spectrum.get('m/z array', []), 3),
                                'ints': spectrum.get('intensity array', [])
                            })
        else:
            print("⚡ Using cached .mzML data from RAM...")

        print("Aggregating full spectral data for all peak windows...")
        peak_full_spectra = {row['peak_id']: {} for _, row in detected_df.iterrows()}

        for scan in global_mzml_cache:
            rt = scan['rt']
            for _, row in detected_df.iterrows():
                if not pd.isna(row['peak start time']) and row['peak start time'] <= rt <= row['peak end time']:
                    pid = row['peak_id']
                    for m, intensity in zip(scan['mzs'], scan['ints']):
                        peak_full_spectra[pid][m] = peak_full_spectra[pid].get(m, 0) + intensity

        df_tree_eval = df_tree.copy()
        df_tree_eval['Is_Intact'] = ~df_tree_eval['Peak Name'].astype(str).str.contains('Frag')

        tree_mzs = df_tree_eval['m/z'].values
        tree_targets = df_tree_eval['Target number'].astype(str).values
        tree_smiles = df_tree_eval['SMILES'].values
        tree_names = df_tree_eval['Peak Name'].values
        tree_is_intact = df_tree_eval['Is_Intact'].values
        tree_abunds = df_tree_eval['Relative Abundance (%)'].values

        # 3. Build Peak Tasks
        pending_peaks = []
        for _, peak_row in detected_df.iterrows():
            pid = peak_row['peak_id']
            raw_spectrum = peak_full_spectra.get(pid, {})

            filtered_spectrum = {}
            if raw_spectrum:
                max_int = max(raw_spectrum.values())
                filtered_spectrum = {m: inten for m, inten in raw_spectrum.items() if inten >= max_int * NOISE_THRESHOLD}

            task = {
                "Peak #": pid,
                "Retention Time (min)": f"{peak_row['retention_time']:.2f}",
                "Spectrum": filtered_spectrum
            }
            pending_peaks.append(task)

        # 4. Worker Function
        def process_peak(task):
            peak_num = task['Peak #']
            spectrum = task['Spectrum']
            out_rows = []

            if not spectrum:
                task.update({
                    'Calculated m/z': "N/A", 'Observed m/z': "N/A", 'Mass Difference': "N/A", 'Identified Adduct': "No MS Signal",
                    'Natural Abundance (%)': "N/A", 'Verification Status': "SKIPPED", 'Agent Rationale': "No MS signal."
                })
                return [task]

            obs_mzs = np.array(list(spectrum.keys()))
            obs_ints = np.array(list(spectrum.values()))
            max_intensity = obs_ints.max()

            intact_matches = []
            frag_matches = []

            for i, obs_mz in enumerate(obs_mzs):
                diffs = np.abs(tree_mzs - obs_mz)
                best_idx = np.argmin(diffs)

                if diffs[best_idx] <= TOLERANCE_DA:
                    match_data = {
                        'mz': obs_mz,
                        'calc_mz': tree_mzs[best_idx],
                        'intensity': obs_ints[i],
                        'target': tree_targets[best_idx],
                        'smiles': tree_smiles[best_idx],
                        'name': tree_names[best_idx],
                        'is_intact': tree_is_intact[best_idx],
                        'diff': diffs[best_idx],
                        'actual_diff': obs_mz - tree_mzs[best_idx],
                        'abund': tree_abunds[best_idx]
                    }

                    if tree_is_intact[best_idx]:
                        intact_matches.append(match_data)
                    else:
                        frag_matches.append(match_data)

            target_scores = {}
            target_evidence = {}

            for match in intact_matches:
                t = match['target']
                if t not in target_scores:
                    target_scores[t] = 0
                    target_evidence[t] = []

                target_evidence[t].append(match)
                rel_int = match['intensity'] / max_intensity
                target_scores[t] += (100 * rel_int)

                if '[M+H]+' in match['name'] or '[M-H]-' in match['name'] or '[M+Na]+' in match['name']:
                    target_scores[t] += (500 * rel_int)

            if target_scores:
                for match in frag_matches:
                    t = match['target']
                    if t in target_scores:
                        target_evidence[t].append(match)
                        rel_int = match['intensity'] / max_intensity
                        target_scores[t] += (5 * rel_int)
            else:
                for match in frag_matches:
                    t = match['target']
                    if t not in target_scores:
                        target_scores[t] = 0
                        target_evidence[t] = []
                    target_evidence[t].append(match)
                    rel_int = match['intensity'] / max_intensity
                    target_scores[t] += (10 * rel_int)

            if not target_scores:
                top_3_mzs = sorted(spectrum.items(), key=lambda x: x[1], reverse=True)[:3]
                mz_str = ", ".join([str(m) for m, _ in top_3_mzs])

                prompt = f"""
                You are an expert analytical chemist.
                An LC-MS peak has the following dominant m/z values: {mz_str}
                No library match was found. Propose a single chemical contaminant or background ion class.
                Return EXACTLY two lines:
                Line 1: Proposed identity
                Line 2: A one-sentence explanation.
                """
                try:
                    res = client.models.generate_content(model=active_model, contents=prompt, config=search_config)
                    lines = [line.strip() for line in res.text.strip().split('\n') if line.strip()]
                    adduct = lines[0].replace('`', '') if len(lines) > 0 else "Unknown Contaminant"
                    rationale = lines[1] if len(lines) > 1 else ""
                except Exception as e:
                    if "503 UNAVAILABLE" in str(e):
                        adduct, rationale = "ERROR", "API unavailable due to high demand. Please try re-running the cell."
                    else:
                        adduct, rationale = "Unknown Contaminant", str(e)

                top_mz = top_3_mzs[0][0]
                row = task.copy()
                row.update({
                    "Calculated m/z": "N/A",
                    "Observed m/z": float(top_mz),
                    "Mass Difference": "N/A",
                    "Identified Adduct": f"{adduct} (AI)",
                    "Natural Abundance (%)": "N/A",
                    "Verification Status": "AI PROPOSED",
                    "Agent Rationale": rationale
                })
                row.pop("Spectrum", None)
                out_rows.append(row)
            else:
                winning_target = max(target_scores, key=target_scores.get)
                evidence = target_evidence[winning_target]

                unique_evidence = {}
                for ev in evidence:
                    mz = ev['mz']
                    if mz not in unique_evidence or abs(ev['actual_diff']) < abs(unique_evidence[mz]['actual_diff']):
                        unique_evidence[mz] = ev

                final_evidence = list(unique_evidence.values())
                win_smiles = final_evidence[0]['smiles']

                anchors = [e for e in final_evidence if e['is_intact']]
                fragments = [e for e in final_evidence if not e['is_intact']]

                final_evidence = anchors + fragments
                ev_max_int = max(e['intensity'] for e in final_evidence) or 1.0

                def _ion_line(idx, e):
                    return (f"- ION {idx}: m/z {e['mz']}: {e['name']} (diff: {e['actual_diff']:.4f}, "
                            f"relative intensity: {100 * e['intensity'] / ev_max_int:.1f}% of strongest assigned ion, "
                            f"{100 * e['intensity'] / max_intensity:.1f}% of base peak, "
                            f"theoretical natural abundance: {e['abund']}%)")

                anchor_str = "\n".join([_ion_line(i + 1, e) for i, e in enumerate(anchors)]) or "None detected"
                frag_str = "\n".join([_ion_line(len(anchors) + i + 1, e) for i, e in enumerate(fragments)]) or "None detected"
                n_ions = len(final_evidence)

                prompt = f"""
                You are an expert mass spectrometrist verifying a single species assignment for an LC-MS peak.

                Target Molecule SMILES: {win_smiles}

                Primary Intact Anchor Ions Found (Non-Fragmented Adducts):
                {anchor_str}

                Supporting Fragment Adducts Found:
                {frag_str}

                INSTRUCTIONS:
                1. PRIORITIZE the intact, non-fragmented adducts (e.g., [M+H]+, [M-H]-, [M+Na]+) listed under "Primary Intact Anchor Ions Found".
                2. If a valid, chemically logical intact parent adduct is present, you MUST verify the assignment.
                3. Evaluate the fragmented ion adducts ONLY as secondary evidence to back up the intact assignment.
                4. Do NOT reject an assignment because some fragments are complex or rare, provided the intact parent mass anchors the identity.

                5. For EACH ion individually, act as an expert mass spectrometrist and write a rationale SPECIFIC to that ion's adduct/isotopologue type (e.g. [M+H]+ vs [M+Na]+ vs [M+H]+ (+1 13C) must have different explanations). Comment on:
                   - Its observed relative intensity versus what is expected (e.g. the 13C isotope intensity relative to its monoisotopic parent should roughly match the theoretical natural abundance; sodium adducts are typically weaker than protonated ions in acidic mobile phases unless Na+ is abundant).
                   - The probability/plausibility of the assignment given the other ions present. Isotopologues or minor adducts found WITHOUT their monoisotopic parent (e.g. [M+H]+ (+1 17O) without [M+H]+), or with intensities far exceeding their theoretical natural abundance, are improbable and should be flagged as likely coincidental matches or interferences.
                   - Its mass error.

                Return EXACTLY {n_ions + 2} lines:
                Line 1: ONLY the word "VERIFIED" or "REJECTED".
                Line 2: A brief overall rationale for the assignment.
                Then one line per ion, in order, formatted exactly as "ION <n>: <ion-specific rationale incl. a probability of High/Medium/Low>" for n = 1 to {n_ions}.
                """
                ion_rationales = {}
                try:
                    res = client.models.generate_content(model=active_model, contents=prompt, config=search_config)
                    lines = [line.strip() for line in res.text.strip().split('\n') if line.strip()]
                    status = lines[0].replace('*', '').upper() if len(lines) > 0 else "REJECTED"
                    rationale = lines[1] if len(lines) > 1 else "No explanation provided."
                    for line in lines[2:]:
                        m = re.match(r"^[\-\*\s]*ION\s*(\d+)\s*[:\-]\s*(.+)$", line.replace('**', ''), re.IGNORECASE)
                        if m:
                            ion_rationales[int(m.group(1))] = m.group(2).strip()
                except Exception as e:
                    if "503 UNAVAILABLE" in str(e):
                        status, rationale = "ERROR", "API unavailable due to high demand. Please try re-running the cell."
                    else:
                        status, rationale = "ERROR", str(e)

                for ion_idx, e in enumerate(final_evidence, start=1):
                    row = task.copy()
                    row.update({
                        "Calculated m/z": f"{e['calc_mz']:.4f}",
                        "Observed m/z": e['mz'],
                        "Mass Difference": f"{e['actual_diff']:.4f}",
                        "Identified Adduct": e['name'],
                        "Natural Abundance (%)": e['abund'],
                        "Verification Status": status,
                        "Agent Rationale": ion_rationales.get(ion_idx, f"[Hierarchical Profile] {rationale}")
                    })
                    row.pop("Spectrum", None)
                    out_rows.append(row)

            return out_rows

        # 5. Execute Pipeline in Parallel
        print(f"🚀 Assigning and verifying {len(pending_peaks)} holisitic peak clusters (Workers: {MAX_WORKERS})...\n")

        processed_rows = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [executor.submit(process_peak, task) for task in pending_peaks]
            for future in concurrent.futures.as_completed(futures):
                res_list = future.result()
                processed_rows.extend(res_list)

                if res_list:
                    peak_id = res_list[0]['Peak #']
                    status = res_list[0].get('Verification Status', 'SKIPPED')
                    icon = "✅" if status == "VERIFIED" else "❌" if "REJECTED" in status else "⏭️"
                    print(f"{icon} Peak {peak_id} Cluster Evaluated: [{status}]")

        exec_time = time.time() - start_time
        print(f"\n✅ Verification & Refinement complete in {exec_time:.2f} seconds.\n")

        # 6. Build Final assigned_df
        assigned_df = pd.DataFrame(processed_rows)

        assigned_df['is_intact'] = ~assigned_df['Identified Adduct'].astype(str).str.contains('Frag')
        assigned_df = assigned_df.sort_values(by=['Peak #', 'is_intact', 'Observed m/z'], ascending=[True, False, True]).drop(columns=['is_intact']).reset_index(drop=True)

        base_cols = ['Peak #', 'Retention Time (min)', 'Calculated m/z', 'Observed m/z', 'Mass Difference', 'Identified Adduct', 'Natural Abundance (%)', 'Verification Status', 'Agent Rationale']
        assigned_df = assigned_df[base_cols]

        display(Markdown("### 🕵️ Peak-Level Profile Verification Summary"))
        summary_df = assigned_df.drop_duplicates(subset=['Peak #'])
        for _, row in summary_df.iterrows():
            if row['Verification Status'] != "SKIPPED":
                display(Markdown(f"**Peak {row['Peak #']} (RT {row['Retention Time (min)']})** — **{row['Verification Status']}**<br>*{row['Agent Rationale']}*"))

        display(Markdown("<hr>"))
        display(Markdown("### 📊 Final assigned_df"))

        def highlight_assigned_peaks(row):
            colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
            c = mcolors.to_rgba(colors[(int(row['Peak #']) - 1) % len(colors)], alpha=0.4)
            bg_color = f'rgba({int(c[0]*255)}, {int(c[1]*255)}, {int(c[2]*255)}, {c[3]})'
            return [f'background-color: {bg_color}' for _ in row]

        display(assigned_df.style.apply(highlight_assigned_peaks, axis=1))

        # -------------------------------------------------------------------------------------
        # 7. GENERATE LABELED HPLC CHROMATOGRAM
        # -------------------------------------------------------------------------------------
        display(Markdown("<hr>"))
        display(Markdown("### 🗺️ Fully Labeled HPLC Chromatogram"))

        if 'hplc-py' not in sys.path:
            sys.path.append(os.path.abspath('hplc-py'))
        from hplc.quant import Chromatogram

        chrom_plot = Chromatogram(hplc_df, cols={'time':'Time', 'signal':'Signal'}, time_window=[200, 300])
        prom_val = prominence_val if 'prominence_val' in globals() else 0.05
        _ = chrom_plot.fit_peaks(prominence=prom_val, buffer=0)

        fig1, ax1 = chrom_plot.show()
        fig1.set_size_inches(26, 12)

        ymin_initial, _ = ax1.get_ylim()
        lines_ydata = [line.get_ydata() for line in ax1.get_lines()]
        max_plotted_signal = max([np.max(y) for y in lines_ydata]) if lines_ydata else hplc_df['Signal'].max()

        signal_raw = hplc_df['Signal'].values
        time_arr = hplc_df['Time'].values
        signal_norm = (signal_raw - signal_raw.min()) / (signal_raw.max() - signal_raw.min())
        peak_indices, _ = scipy.signal.find_peaks(signal_norm, prominence=0.01)

        verified_peaks = assigned_df[assigned_df['Verification Status'] == 'VERIFIED']
        unique_peaks = verified_peaks.drop_duplicates(subset=['Peak #'])

        placed_boxes = []

        if 'detected_df' in globals():
            for _, det_row in detected_df.iterrows():
                peak_num = det_row['peak_id']
                rt_deconv = det_row['retention_time']

                closest_idx = (np.abs(time_arr - rt_deconv)).argmin()
                rt_anchor = time_arr[closest_idx]

                raw_line = ax1.lines[0]
                plotted_x = raw_line.get_xdata()
                plotted_y = raw_line.get_ydata()
                plot_closest_idx = (np.abs(plotted_x - rt_anchor)).argmin()
                peak_val = plotted_y[plot_closest_idx]

                match = unique_peaks[unique_peaks['Peak #'] == peak_num]
                if not match.empty:
                    adduct_str = str(match.iloc[0]['Identified Adduct'])
                    t_match = re.search(r'T\((.*?)\)', adduct_str)

                    if t_match:
                        target_str = t_match.group(1)
                        comp_name = f"Target {target_str}"

                        intact_tree_rows = df_tree[
                            (df_tree['Target number'].astype(str) == target_str) &
                            (~df_tree['Peak Name'].astype(str).str.contains('Frag'))
                        ]

                        if not intact_tree_rows.empty:
                            smiles = intact_tree_rows.iloc[0]['SMILES']

                            if pd.notna(smiles):
                                mol = Chem.MolFromSmiles(smiles)
                                if mol:
                                    y_box = peak_val + max_plotted_signal * 0.40
                                    x_box = rt_anchor

                                    for bx, by in placed_boxes:
                                        if abs(bx - x_box) < 10:
                                            y_box = max(y_box, by + max_plotted_signal * 0.55)

                                    placed_boxes.append((x_box, y_box))

                                    img = Draw.MolToImage(mol, size=(100, 100))
                                    imagebox = OffsetImage(img, zoom=0.6)
                                    imagebox.image.axes = ax1

                                    ab = AnnotationBbox(imagebox, (rt_anchor, peak_val),
                                                        xybox=(x_box, y_box),
                                                        xycoords='data',
                                                        boxcoords="data",
                                                        pad=0.3,
                                                        arrowprops=dict(arrowstyle="->", color='black', alpha=0.8))
                                    ax1.add_artist(ab)

                                    i_add = re.sub(r'\s*\(Target [0-9, ]+\)$', '', adduct_str)
                                    o_mz = match.iloc[0]['Observed m/z']
                                    norm_area = det_row.get('normalised area', 'N/A')

                                    img_label = f"Peak {peak_num}\n{comp_name}\n{i_add}\nm/z {o_mz}\nArea: {norm_area}"

                                    ax1.text(x_box, y_box + max_plotted_signal * 0.18, img_label,
                                            ha='center', va='bottom', fontsize=8, color='black',
                                            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7))

        if placed_boxes:
            max_y_used = max([b[1] for b in placed_boxes]) + max_plotted_signal * 0.55
        else:
            max_y_used = max_plotted_signal * 1.5

        plot_top = max(max_y_used, max_plotted_signal * 1.6)
        ax1.set_ylim(ymin_initial, plot_top)

        line_top = plot_top * 0.94
        text_top = plot_top * 0.95

        for idx in peak_indices:
            rt_topo = time_arr[idx]
            ax1.vlines(rt_topo, ymin_initial, line_top, linestyle='--', color='dodgerblue', alpha=0.5, zorder=0)
            ax1.text(rt_topo, text_top, f'{int(round(rt_topo))}', ha='center', va='bottom', fontsize=9, color='dodgerblue', fontweight='bold')

        ax1.set_title('HPLC Chromatogram Peak Detection and Deconvolution with Identified Molecules', fontsize=16)
        ax1.set_xlabel('Retention Time (seconds)', fontsize=14)
        ax1.set_xticks(np.arange(200, 301, 10))
        ax1.set_ylabel('Total Ion Count (baseline corrected)', fontsize=14)

        handles, labels = ax1.get_legend_handles_labels()
        new_labels = []
        for lbl in labels:
            if lbl.lower().startswith('peak '):
                try:
                    p_num = int(lbl.split()[1])
                    m_df = assigned_df[(assigned_df['Peak #'] == p_num) & (assigned_df['Verification Status'] == 'VERIFIED')]
                    if not m_df.empty:
                        adduct_str = str(m_df.iloc[0]['Identified Adduct'])
                        t_match = re.search(r'T\((.*?)\)', adduct_str)
                        c_name = f"Target {t_match.group(1)}" if t_match else "Unknown"

                        o_mz = m_df.iloc[0]['Observed m/z']
                        i_add = re.sub(r'\s*\(Target [0-9, ]+\)$', '', adduct_str)
                        rt_val = m_df.iloc[0]['Retention Time (min)']
                        new_labels.append(f"Peak {p_num}: {c_name} (RT {rt_val}s, {i_add}, m/z {o_mz})")
                    else:
                        new_labels.append(lbl)
                except (ValueError, IndexError):
                    new_labels.append(lbl)
            else:
                new_labels.append(lbl)

        from matplotlib.lines import Line2D
        handles.append(Line2D([0], [0], color='dodgerblue', linestyle='--', lw=1.5))
        new_labels.append('Topographical Peaks')

        ax1.legend(handles, new_labels, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11)
        plt.tight_layout()
        plt.show()

        # -------------------------------------------------------------------------------------
        # 8. GENERATE MASS SPECTRA PLOTS
        # -------------------------------------------------------------------------------------
        display(Markdown("<hr>"))
        display(Markdown("### 📈 Peak Mass Spectra & Structural Assignments"))

        target_rts = detected_df['retention_time'].values
        peak_ids = detected_df['peak_id'].values
        num_peaks = len(target_rts)

        fig, axes = plt.subplots(
            num_peaks, 3,
            figsize=(22, 6.5 * num_peaks),
            gridspec_kw={'width_ratios': [3.5, 1.3, 3.2]}
        )

        if num_peaks == 1:
            axes = np.array([axes])

        for i, target_rt in enumerate(target_rts):
            ax_ms = axes[i, 0]
            ax_mol = axes[i, 1]
            ax_tbl = axes[i, 2]

            current_peak_id = int(peak_ids[i])

            raw_spec = peak_full_spectra.get(current_peak_id, {})
            if raw_spec:
                mzs = np.array(list(raw_spec.keys()))
                ints = np.array(list(raw_spec.values()))
                sort_idx = np.argsort(mzs)
                mz = mzs[sort_idx]
                intensity = ints[sort_idx]
            else:
                mz = np.array([])
                intensity = np.array([])

            current_assignments = assigned_df[assigned_df['Peak #'] == current_peak_id].copy()

            # --- COLUMN 1: Mass Spectrum Plot ---
            peak_color = f'C{i % 10}'

            if len(mz) > 0:
                ax_ms.vlines(mz, ymin=0, ymax=intensity, color=peak_color, linewidth=1.2)

                if len(intensity) > 0:
                    y_max = intensity.max()
                    ax_ms.set_ylim(0, y_max * 1.25)

                    sorted_idx = np.argsort(intensity)[::-1]
                    placed_x = []
                    for idx in sorted_idx:
                        x, y = mz[idx], intensity[idx]
                        if y < 0.02 * y_max:
                            continue

                        if not any(abs(x - px) < 0.6 for px in placed_x) and len(placed_x) < 10:
                            ax_ms.text(x, y + 0.02 * y_max, f'{x:.4f}', fontsize=7.5,
                                       ha='center', va='bottom', rotation=90, color='#2c3e50')
                            placed_x.append(x)

                ax_ms.set_title(f'Peak {current_peak_id} Aggregated MS (Target RT: {target_rt:.2f} s)', fontsize=10, weight='bold')
                ax_ms.set_xlabel('m/z', fontsize=9)
                ax_ms.set_ylabel('Aggregated Intensity', fontsize=9)
                ax_ms.grid(True, linestyle='--', alpha=0.3)
            else:
                ax_ms.set_title(f'No spectrum found for Peak {current_peak_id}')
                ax_ms.axis('off')

            # --- COLUMN 2: RDKit Structure (Intact Prioritization) ---
            ax_mol.axis('off')
            smiles = None
            comp_name = "Unknown"
            exact_mass_str = "N/A"

            if not current_assignments.empty and 'df_tree' in globals():
                verified = current_assignments[current_assignments['Verification Status'] == 'VERIFIED']
                if not verified.empty:
                    reference_row = verified.iloc[0]

                    adduct_str = str(reference_row['Identified Adduct'])
                    t_match = re.search(r'T\((.*?)\)', adduct_str)

                    if t_match:
                        target_str = t_match.group(1)

                        intact_tree_rows = df_tree[
                            (df_tree['Target number'].astype(str) == target_str) &
                            (~df_tree['Peak Name'].astype(str).str.contains('Frag'))
                        ]

                        if not intact_tree_rows.empty:
                            smiles = intact_tree_rows.iloc[0]['SMILES']
                            comp_name = f"Target {target_str}"

                    if not smiles:
                        obs_mz = pd.to_numeric(reference_row['Observed m/z'], errors='coerce')
                        if pd.notna(obs_mz):
                            match_df = df_tree.copy()
                            match_df['diff'] = np.abs(match_df['m/z'] - obs_mz)
                            best_match = match_df.sort_values('diff').iloc[0]

                            if best_match['diff'] <= 0.05:
                                smiles = best_match.get('SMILES', None)
                                target_val = best_match.get('Target number', 'Unknown')
                                if str(target_val) != 'Unknown':
                                    comp_name = f"Target {target_val}"

                if smiles and smiles != 'Unknown':
                    mol = Chem.MolFromSmiles(smiles)
                    if mol:
                        exact_mass_val = Descriptors.ExactMolWt(mol)
                        exact_mass_str = f"{exact_mass_val:.4f} Da"
                        img = Draw.MolToImage(mol, size=(220, 220))
                        ax_mol.imshow(img)

                        wrapped_title = textwrap.fill(comp_name, width=25)
                        ax_mol.set_title(f"{wrapped_title}\nExact Mass: {exact_mass_str}", fontsize=10, weight='bold', pad=8)
                    else:
                        ax_mol.text(0.5, 0.5, f'{comp_name}\nInvalid SMILES', ha='center', va='center', wrap=True)
                else:
                    if current_assignments.empty:
                        ax_mol.text(0.5, 0.5, 'No Assignments', ha='center', va='center')
                    else:
                        ax_mol.text(0.5, 0.5, 'Unmapped Structure', ha='center', va='center')
            else:
                ax_mol.text(0.5, 0.5, 'No Compound Mapped', ha='center', va='center')

            # --- COLUMN 3: Mini-Table of Adduct Assignments ---
            ax_tbl.axis('off')
            col_labels = ['Obs m/z', 'Calc m/z', 'Mass Diff', 'Identified Adduct', 'Abund.']
            table_data = []

            if not current_assignments.empty:
                for _, row in current_assignments.iterrows():
                    obs_v = str(row.get('Observed m/z', 'N/A'))
                    calc_v = str(row.get('Calculated m/z', 'N/A'))
                    diff = str(row.get('Mass Difference', 'N/A'))
                    adduct = str(row.get('Identified Adduct', 'N/A'))

                    adduct = re.sub(r'\s*\(Target [0-9, ]+\)$', '', adduct)
                    abund = str(row.get('Natural Abundance (%)', 'N/A'))
                    adduct = textwrap.fill(adduct, width=28)

                    if calc_v != 'N/A': calc_v = f"{float(calc_v):.4f}"
                    if obs_v != 'N/A': obs_v = f"{float(obs_v):.4f}"

                    if abund != 'N/A' and abund != 'Predicted' and not abund.endswith('%'):
                        try:
                            abund = f"{float(abund):.1f}%"
                        except ValueError:
                            pass

                    table_data.append([obs_v, calc_v, diff, adduct, abund])

                MAX_ROWS = 10
                if len(table_data) > MAX_ROWS:
                    excess_count = len(table_data) - MAX_ROWS
                    table_data = table_data[:MAX_ROWS]
                    table_data.append(['...', '...', '...', f'+ {excess_count} more supporting ions', '...'])
            else:
                table_data.append(['N/A', 'N/A', 'N/A', 'No MS Signal / Unassigned', 'N/A'])

            line_counts = [1]
            for row in table_data:
                line_counts.append(max([str(item).count('\n') + 1 for item in row]))

            tbl = ax_tbl.table(
                cellText=table_data,
                colLabels=col_labels,
                colWidths=[0.15, 0.16, 0.15, 0.40, 0.14],
                loc='center',
                cellLoc='center'
            )
            tbl.auto_set_font_size(False)
            tbl.set_fontsize(8)

            header_color = mcolors.to_hex(plt.rcParams['axes.prop_cycle'].by_key()['color'][i % 10])
            base_h = 0.045

            for (r_idx, c_idx), cell in tbl.get_celld().items():
                cell.set_height(base_h * line_counts[r_idx])

                if r_idx == 0:
                    cell.set_text_props(weight='bold', color='white')
                    cell.set_facecolor(header_color)
                else:
                    cell.set_facecolor('#f8f9fa' if r_idx % 2 == 0 else '#ffffff')

            total_height = sum(line_counts) * base_h
            y_title = 0.5 + (total_height / 2) + 0.02

            ax_tbl.text(0.5, y_title, f'Primary Anchor Assignment (Peak {current_peak_id})',
                        ha='center', va='bottom', fontsize=9.5, weight='bold', color='#2c3e50')

        plt.tight_layout(pad=3.0)
        plt.subplots_adjust(wspace=0.20, hspace=0.45)
        plt.show()

    except Exception as e:
        print(f"Refinement pipeline error: {e}")

In [ ]:
#@title Automated Structural Verification (PubChem & NCI CACTUS)

#Preserves 3D stereochemical data from the RDKit fragmentation tree to API query PubChem and NCI CACTUS
import requests
import pandas as pd
import time
import re
import urllib.parse
from rdkit import Chem
from IPython.display import display, HTML, Markdown

if 'assigned_df' not in globals() or 'df_tree' not in globals():
    display(HTML("<b style='color:red;'>⚠️ 'assigned_df' or 'df_tree' not found. Please run the AI assignment cell first.</b>"))
else:
    print("🔍 Generating stereo-specific InChIKeys to query PubChem's ranked synonym index...")

    ground_truth_results = []

    verified_peaks = assigned_df[assigned_df['Verification Status'] == 'VERIFIED'].copy()
    is_primary = verified_peaks['Identified Adduct'].astype(str).str.contains(r'\[M\+H\]\+|\[M-H\]\-|\[M\+Na\]\+', regex=True)
    is_isotope = verified_peaks['Identified Adduct'].astype(str).str.contains(r'\(\+', regex=True)
    verified_peaks = verified_peaks[is_primary & ~is_isotope].drop_duplicates(subset=['Peak #'])
    intact_tree = df_tree[~df_tree['Peak Name'].astype(str).str.contains('Frag')]

    def purify_molecule(smiles):
        """Scrub isotopes and explicit Hs, but PRESERVE stereochemistry for precise biological matching."""
        mol = Chem.MolFromSmiles(smiles)
        if not mol: return None
        for atom in mol.GetAtoms():
            atom.SetIsotope(0)
        mol = Chem.RemoveHs(mol)
        # We NO LONGER remove stereochemistry so we can differentiate Adenosine from Xylosyladenine
        return mol

    def get_best_pubchem_synonym(synonyms):
        valid_names = []
        for syn in synonyms:
            if len(syn) < 5 or len(syn) > 35:
                continue
            if re.search(r'[\(\)\[\]\.\,]', syn):
                continue
            if re.match(r'^[A-Za-z0-9\-\']{4,12}$', syn) and sum(c.isdigit() for c in syn) > 1:
                continue
            if syn.lower().startswith('poly'):
                continue
            if sum(c.isdigit() for c in syn) > 2 or syn.count('-') > 2:
                continue
            if sum(1 for c in syn if c.isupper()) > len(syn) / 2:
                continue
            valid_names.append(syn)

        if not valid_names:
            return None

        biochem_suffixes = ('osine', 'idine', 'ol', 'ate', 'in', 'acid', 'one', 'gen')
        tier1 = [n for n in valid_names if n.lower().endswith(biochem_suffixes)]
        if tier1:
            return tier1[0].title()

        tier2 = [n for n in valid_names if n.replace(' ', '').replace('-', '').isalpha()]
        if tier2:
            return tier2[0].title()

        return valid_names[0].title()

    for _, row in verified_peaks.iterrows():
        peak_id = row['Peak #']
        obs_mz = row['Observed m/z']
        ai_adduct = str(row['Identified Adduct'])

        t_match = re.search(r'T\(([0-9]+)\)', ai_adduct)
        if not t_match: continue
        target_num = t_match.group(1)

        target_row = intact_tree[intact_tree['Target number'].astype(str) == target_num]
        if target_row.empty: continue

        raw_smiles = target_row.iloc[0]['SMILES']
        pure_mol = purify_molecule(raw_smiles)
        if not pure_mol: continue

        print(f"Querying Peak {peak_id} -> Target {target_num}...")

        common_name = "N/A"
        nci_iupac = "No Match"

        # 1. Generate the exact stereo-specific InChIKey directly via RDKit
        std_inchikey = Chem.MolToInchiKey(pure_mol)

        # 2. Query PubChem's Synonym endpoint using the exact biological InChIKey
        if std_inchikey:
            try:
                pc_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{std_inchikey}/synonyms/JSON"
                pc_res = requests.get(pc_url, timeout=10)
                if pc_res.status_code == 200:
                    data = pc_res.json()
                    synonyms = data.get('InformationList', {}).get('Information', [{}])[0].get('Synonym', [])
                    best_name = get_best_pubchem_synonym(synonyms)
                    if best_name:
                        common_name = best_name
            except Exception:
                pass

        # 3. Ask CACTUS for the official IUPAC name using the stereo-specific SMILES
        try:
            pure_smiles = Chem.MolToSmiles(pure_mol, isomericSmiles=True)
            nci_url = f"https://cactus.nci.nih.gov/chemical/structure/{urllib.parse.quote(pure_smiles)}/iupac_name"
            nci_res = requests.get(nci_url, timeout=10)
            if nci_res.status_code == 200 and nci_res.text:
                nci_iupac = nci_res.text.strip().split('\n')[0]
        except Exception:
            nci_iupac = "API Error"

        if common_name == "N/A" and nci_iupac not in ["No Match", "API Error"]:
            common_name = nci_iupac

        ground_truth_results.append({
            'Peak #': peak_id,
            'Observed m/z': f"{float(obs_mz):.4f}",
            'AI Target': f"Target {target_num}",
            'Common Name': common_name,
            'IUPAC Name (NCI)': nci_iupac
        })

        time.sleep(0.5)

    if ground_truth_results:
        gt_df = pd.DataFrame(ground_truth_results)
        display(Markdown("### 🏛️ Structural Verification (Stereo-Specific Biological Resolution)"))

        def highlight_hits(row):
            colors = []
            for col in gt_df.columns:
                if col == 'Common Name' and row[col] not in ["N/A"]:
                    colors.append('background-color: #fff9c4; font-weight: bold;')
                elif col == 'IUPAC Name (NCI)' and row[col] not in ["No Match", "API Error"]:
                    colors.append('background-color: #e3f2fd')
                else:
                    colors.append('')
            return colors

        display(gt_df.style.apply(highlight_hits, axis=1).set_properties(**{'text-align': 'left'}))
    else:
        print("No valid SMILES targets found to query.")

In [ ]:
#@title 💬 Discuss your LC-MS Results with Gemini
import ipywidgets as w
from IPython.display import display, HTML, clear_output
import pandas as pd
import getpass
import re
from google import genai
from google.genai import types

# 1. Verify prerequisites
if 'assigned_df' not in globals() or 'detected_df' not in globals():
    display(HTML("<b style='color:red;'>⚠️ Please run the Peak Detection and Assignment cells first.</b>"))
else:
    if 'gemini_api_key' not in globals() or not gemini_api_key:
        gemini_api_key = getpass.getpass("Enter your Gemini API Key: ")

    try:
        client = genai.Client(api_key=gemini_api_key)

        # Select Model
        available_models = [m.name.replace('models/', '') for m in client.models.list()]
        flash_models = [name for name in available_models if re.match(r'^gemini-\d+\.\d+-flash$', name)]
        flash_models.sort()
        chat_model = flash_models[-1] if flash_models else 'gemini-2.5-flash'

        # 2. Compile notebook context for the AI
        summary_lines = ["--- CURRENT LC-MS RUN DATA ---"]

        # Aggregate Peak Data
        unique_peaks = assigned_df.drop_duplicates(subset=['Peak #'])
        for _, row in unique_peaks.iterrows():
            peak_num = row['Peak #']
            rt = row['Retention Time (min)']
            status = row['Verification Status']
            adduct = row['Identified Adduct']
            rationale = row['Agent Rationale']

            # Fetch normalizer area if available
            area = "Unknown"
            if 'detected_df' in globals():
                det_match = detected_df[detected_df['peak_id'] == peak_num]
                if not det_match.empty:
                    area = det_match.iloc[0].get('normalised area', 'Unknown')

            summary_lines.append(f"Peak {peak_num} (RT: {rt}s, Area: {area}): [{status}] Assigned to {adduct}. Rationale: {rationale}")

        system_context = "\n".join(summary_lines)

        system_instruction = f"""
        You are an expert analytical chemist and mass spectrometry assistant.
        The user is analyzing an HPLC-MS run. Here is the verified data for the current chromatogram:

        {system_context}

        Answer the user's questions based strictly on this dataset.
        If they ask for summaries, pinpoint specific peaks, retention times, and molecular targets.
        Format your responses cleanly using bullet points or short paragraphs. Avoid repeating the raw data verbatim unless asked.
        IMPORTANT: Do not use LaTeX formatting (e.g., $ signs) for chemical formulas, adducts, or m/z values. Use standard plain text formatting (e.g., [M+H]+, m/z 269.088).
        """

        # 3. Initialize Chat Session
        chat_session = client.chats.create(
            model=chat_model,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3
            )
        )

        # 4. Build UI
        # Use overflow='auto' instead of overflow_y to ensure compatibility
        chat_output = w.Output(layout=w.Layout(height='400px', overflow='auto', border='1px solid #ccc', padding='10px', display='block'))
        user_input = w.Textarea(placeholder='Ask Gemini about your chromatogram...', layout=w.Layout(width='85%', height='60px'))
        send_btn = w.Button(description='Send', button_style='primary', layout=w.Layout(width='14%', height='60px'))

        # Quick-prompt buttons
        btn_summary = w.Button(description='Summarize Findings', button_style='info', layout=w.Layout(width='auto'))
        btn_rejects = w.Button(description='Analyze Rejected Peaks', button_style='warning', layout=w.Layout(width='auto'))

        preset_box = w.HBox([w.HTML("<b>Quick Prompts:</b> "), btn_summary, btn_rejects], layout=w.Layout(margin='10px 0'))
        input_box = w.HBox([user_input, send_btn], layout=w.Layout(align_items='center', justify_content='space-between'))

        # 5. Interaction Logic
        def append_message(sender, text, color):
            with chat_output:
                display(HTML(f"<div style='margin-bottom:10px;'><b style='color:{color};'>{sender}:</b><br/>{text}</div>"))

        def send_message(text):
            if not text.strip():
                return

            user_input.value = ''
            append_message('You', text, '#2c3e50')

            with chat_output:
                display(HTML("<i id='loading'>Gemini is analyzing...</i>"))

            try:
                response = chat_session.send_message(text)

                # Clear output to remove the loading text, then re-render history manually
                chat_output.clear_output(wait=True)
                for msg in chat_session.get_history():
                    sender_name = 'You' if msg.role == 'user' else 'Gemini'
                    msg_color = '#2c3e50' if msg.role == 'user' else '#27ae60'
                    if msg.parts:
                        # Simple markdown parsing for bold text
                        formatted_text = msg.parts[0].text.replace('\n', '<br/>').replace('**', '<b>').replace('**', '</b>')
                        with chat_output:
                            display(HTML(f"<div style='margin-bottom:10px;'><b style='color:{msg_color};'>{sender_name}:</b><br/>{formatted_text}</div>"))
            except Exception as e:
                with chat_output:
                    display(HTML(f"<div style='color:red;'><b>Error:</b> {str(e)}</div>"))

        def on_send_click(b):
            send_message(user_input.value)

        def on_preset_summary(b):
            send_message("Please provide a concise executive summary of the verified compounds in this run, including their retention times and relative abundance.")

        def on_preset_rejects(b):
            send_message("Which peaks were REJECTED or left unassigned, and what are the most likely chemical reasons for this based on your rationale?")

        send_btn.on_click(on_send_click)
        btn_summary.on_click(on_preset_summary)
        btn_rejects.on_click(on_preset_rejects)

        # Allow pressing Enter (Shift+Enter) to send
        def on_enter(change):
            if '\n' in change.new:
                send_message(change.new.replace('\n', ''))
        user_input.observe(on_enter, names='value')

        # 6. Group all elements into a master VBox to prevent overlapping
        header = w.HTML("<h3>🧪 Talk to Gemini about your LC-MS Run</h3>")

        main_ui_container = w.VBox([
            header,
            chat_output,
            preset_box,
            input_box
        ], layout=w.Layout(width='100%'))

        # Display the single master container
        display(main_ui_container)

        with chat_output:
            display(HTML("<div style='color:#7f8c8d; font-style:italic;'>System context loaded. I'm ready to discuss your assigned LC-MS peaks. Try clicking a quick prompt or typing a question below.</div>"))

    except Exception as e:
        print(f"Failed to initialize Gemini Client: {e}")

In [ ]:
#@title 📄 Export Results to HTML Report
import base64
import io
from IPython.display import HTML, display

def fig_to_base64(figure):
    """Converts a matplotlib figure to a base64 encoded string for HTML embedding."""
    buf = io.BytesIO()
    figure.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')

# Initialize HTML string with styling
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>LC-MS Analysis Report</title>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 40px; color: #333; }
        h1 { color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; }
        h2 { color: #2980b9; margin-top: 30px; }
        table { border-collapse: collapse; width: 100%; margin-bottom: 20px; font-size: 13px; }
        th, td { border: 1px solid #ddd; padding: 10px; text-align: left; }
        th { background-color: #f8f9fa; color: #2c3e50; }
        tr:nth-child(even) { background-color: #fbfcfc; }
        .section { margin-bottom: 50px; }
        img { max-width: 100%; height: auto; border: 1px solid #ddd; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }
        .summary-box { font-family: monospace; background: #f8f9fa; padding: 15px; border: 1px solid #ddd; border-left: 4px solid #3498db; }
    </style>
</head>
<body>
    <h1>Automated LC-MS Analysis Report</h1>
"""

# 1. Chromatogram Plot (from fig1)
if 'fig1' in globals():
    html_content += f"""
    <div class="section">
        <h2>1. Fully Labeled HPLC Chromatogram</h2>
        <img src="data:image/png;base64,{fig_to_base64(fig1)}" alt="Labeled Chromatogram">
    </div>
    """

# 2. Structural Verification Data (from gt_df)
if 'gt_df' in globals():
    html_content += f"""
    <div class="section">
        <h2>2. Structural Verification (PubChem & NCI CACTUS)</h2>
        {gt_df.to_html(classes='table', index=False, justify='left')}
    </div>
    """

# 3. Final Peak Assignments Data (from assigned_df)
if 'assigned_df' in globals():
    html_content += f"""
    <div class="section">
        <h2>3. Final Peak Assignments & Agent Rationale</h2>
        {assigned_df.to_html(classes='table', index=False, justify='left')}
    </div>
    """

# 4. Mass Spectra Plots (from fig)
if 'fig' in globals():
    html_content += f"""
    <div class="section">
        <h2>4. Peak Mass Spectra & Structural Assignments</h2>
        <img src="data:image/png;base64,{fig_to_base64(fig)}" alt="Mass Spectra Grids">
    </div>
    """

# 5. Gemini Analysis Summary Context (from system_context)
if 'system_context' in globals():
    formatted_context = system_context.replace('\n', '<br>')
    html_content += f"""
    <div class="section">
        <h2>5. AI Run Summary Context</h2>
        <div class="summary-box">
            {formatted_context}
        </div>
    </div>
    """

html_content += """
</body>
</html>
"""

# Save to an HTML file
report_filename = 'lcms_analysis_report.html'
with open(report_filename, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"✅ HTML report successfully generated: {report_filename}")

# Provide an interactive download link for Google Colab
def create_download_link(filename, title="📥 Download HTML Report"):
    html = f'<a href="{filename}" download="{filename}" target="_blank" style="display:inline-block; padding:12px 24px; background-color:#2ecc71; color:white; text-decoration:none; border-radius:5px; font-weight:bold; font-family:sans-serif; margin-top:15px; box-shadow: 0 2px 4px rgba(0,0,0,0.2);">{title}</a>'
    return HTML(html)

display(create_download_link(report_filename))